In [0]:
%run ../includes/storage_config

In [0]:
# read data 
services_df = spark.table(processed_table_name_services)
stops_df = spark.table(processed_table_name_stops)


Select fields:
- date 
- on time rate
- cancellation rate
- number of services run
- delayed services
- average delay 
- maximum delay 
- max delay train 

In [0]:
# to do - add service type - add if it's weekday or not
create_daily_summary_sql = f'''
SELECT 
  service_date as date, 
  count(service_rdt_id) as number_of_services,
  round(sum(CASE WHEN service_maximum_delay <= 5 THEN 1 ELSE 0 END) / count(service_rdt_id) * 100, 2) as on_time_rate_5min,
  round(sum(CASE WHEN service_maximum_delay = 0 THEN 1 ELSE 0 END) / count(service_rdt_id) * 100, 2) as on_time_rate_0min,
  round(sum(CASE WHEN service_completely_cancelled THEN 1 ELSE 0 END) / count(service_rdt_id)  * 100 ,2) as cancellation_rate,
  round(sum(CASE WHEN service_partly_cancelled THEN 1 ELSE 0 END) / count(service_rdt_id) * 100,2) as partial_cancellation_rate,
  round(avg(service_maximum_delay),2) as average_max_delay,
  max(service_maximum_delay) as max_delay_mins,
  sum(CASE WHEN service_maximum_delay >= 30 THEN 1 ELSE 0 END) as long_delays_over_30
from {processed_table_name_services}
GROUP BY service_date
ORDER BY service_date;
'''

daily_summary_df = spark.sql(create_daily_summary_sql)

In [0]:
display(daily_summary_df)

In [0]:
# write into delta lake
table_location = f"{schema}.{daily_summary_table_service}"

daily_summary_df.write.mode("overwrite").format("delta").saveAsTable(table_location)